In [1]:
#r "D:/rieckmann/BoSSS/BoSSS-Repos/BoSSS-InterfaceSlip/public/src/L4-application/BoSSSpad/bin/Release/net6.0/BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [79]:
string proj = "SlipConvergence_Droplet_Reference_XNSFE_v3";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


Project name is set to 'SlipConvergence_Droplet_Reference_XNSFE_v3'.
Default Execution queue is chosen for the database.
Creating database '\\dc3\backup\rieckmann\cluster\databases\SlipConvergence_Droplet_Reference_XNSFE_v3'.


In [98]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [99]:
// int[] kelemSeq = new[] {16};//new int[]{4, 8, 16, 32, 64, 128, 256};
int[] kelemSeq = new int[]{4, 8, 16, 32, 64, 128};
var GridSeq = new IGridInfo[kelemSeq.Length];

In [100]:
for(int iGrid = 0; iGrid < GridSeq.Length; iGrid++) {
    
    int kelem = kelemSeq[iGrid];
    
    GridCommons grd;   
    double[] xNodes = GenericBlas.Linspace(-3.0/2.0, 3.0/2.0, 2 * kelem + 1);
    double[] yNodes = GenericBlas.Linspace(0.0, 3.0/2.0, kelem + 1);    
    grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);

    // HIER ANPASSUNG DER RANDBEDINGUNGEN!
    grd.EdgeTagNames.Add(1, "NavierSlip_linear_ConstantHeatflux");
    grd.EdgeTagNames.Add(2, "NavierSlip_linear_ConstantTemperature");


    grd.DefineEdgeTags(delegate (double[] X) {
        byte et = 0;
        if (Math.Abs(X[1] - yNodes.Last()) <= 1.0e-8) // UPPER BOUNDARY
            et = 1;
        if (Math.Abs(X[1] - yNodes.First()) <= 1.0e-8) // LOWER BOUNDARY
            et = 1;
        if (Math.Abs(X[0] - xNodes.First()) <= 1.0e-8) // LEFT BOUNDARY
            et = 2;
        if (Math.Abs(X[0] - xNodes.Last()) <= 1.0e-8) // RIGHT BOUNDARY
            et = 2;
        return et;
    });

    // grd.Name = "StaticDropletOnWall_meshStudy";
    grd.Name = "SlipDropletOnWall_Reference_meshStudy_" + kelem;        
    
    BoSSSshell.WorkflowMgm.DefaultDatabase.SaveGrid(ref grd);
    
    GridSeq[iGrid] = grd;
}

Grid Edge Tags changed.
An equivalent grid (a39c795b-de32-42b5-aa1c-1e092fe0433f) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (bedd01de-183e-4982-9134-1c06e2da5bc0) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (4ecc52f0-973b-4eb6-89b8-9d040a56787c) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (ab719670-954f-4ff2-8fa9-e1c0f8f21d75) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (f96104dd-f98e-44c1-922f-220d9f3af095) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (6e709c1a-9777-4ef6-8092-5fc9e5e019d0) is already present in the database -- the grid will not be saved.


In [101]:
// public static XNSE_Control Droplet_forWorksheet(bool steadyInterface) {

//     XNSE_Control C = new XNSE_Control();

//     //C.DbPath = set by workflowMgm during job creation
//     C.savetodb = true;
//     C.ContinueOnIoError = false;

//     C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.Dropletlike());
//     C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.MovingContactLineLogging());
    
//     // Physical Parameters
//     // ===================
//     C.PhysicalParameters.IncludeConvection = true;
//     C.PhysicalParameters.Material = false;

//     // misc. solver options
//     // ====================
//     C.NonLinearSolver.MaxSolverIterations = 50;
//     C.NonLinearSolver.ConvergenceCriterion = 1e-8;
//     C.LevelSet_ConvergenceCriterion = 1e-6;

//     // Level-Set options (AMR)
//     // =======================
//     C.LSContiProjectionMethod = (steadyInterface) ? BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.ConstrainedDG : BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.ConstrainedDG;
//     C.Option_LevelSetEvolution = (steadyInterface) ? BoSSS.Solution.LevelSetTools.LevelSetEvolution.None : BoSSS.Solution.LevelSetTools.LevelSetEvolution.FastMarching;
//     C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

//     // Timestepping
//     // ============
//     C.TimesteppingMode = AppControl._TimesteppingMode.Transient;

//     C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
//     C.Timestepper_BDFinit = BoSSS.Solution.Timestepping.TimeStepperInit.SingleInit;
//     C.Timestepper_LevelSetHandling = (steadyInterface) ? BoSSS.Solution.XdgTimestepping.LevelSetHandling.None : BoSSS.Solution.XdgTimestepping.LevelSetHandling.Coupled_Once;

//     C.Endtime = 1000;
//     C.saveperiod = 1;       

//     return C;

// }

public static XNSFE_Control Droplet_forWorksheet(bool steadyInterface) {

    XNSFE_Control C = new XNSFE_Control();

    //C.DbPath = set by workflowMgm during job creation
    C.savetodb = true;
    C.ContinueOnIoError = false;

    C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.Dropletlike());
    C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MovingContactLineLogging());
    
    // Physical Parameters
    // ===================
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false;

    // misc. solver options
    // ====================
    C.NonLinearSolver.MaxSolverIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-6;

    // Level-Set options (AMR)
    // =======================
    C.LSContiProjectionMethod = (steadyInterface) ? BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.ConstrainedDG : BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.ConstrainedDG;
    C.Option_LevelSetEvolution = (steadyInterface) ? BoSSS.Solution.LevelSetTools.LevelSetEvolution.None : BoSSS.Solution.LevelSetTools.LevelSetEvolution.FastMarching;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

    // Timestepping
    // ============
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;

    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    C.Timestepper_BDFinit = BoSSS.Solution.Timestepping.TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = (steadyInterface) ? BoSSS.Solution.XdgTimestepping.LevelSetHandling.None : BoSSS.Solution.XdgTimestepping.LevelSetHandling.Coupled_Once;

    C.Endtime = 1000;
    C.saveperiod = 1;       

    return C;

}

In [102]:
int[] degS = new int[] { 2, 3, 4 };

In [103]:
int iDeg0 = 0;
int iGrd0 = 0;

In [131]:
// List<XNSE_Control> controls = new List<XNSE_Control>();
List<XNSFE_Control> controls = new List<XNSFE_Control>();

In [132]:
// Testcase setup
// ==============
bool elliptic  = true;
bool onWall = true;

double radius = 0.8;
double a      = elliptic ? 0.816 : 0.8;
double b      = elliptic ? 0.784 : 0.8;

double[] thetaS            = (new double[] {90}).Select(x => x/180.0 * Math.PI).ToArray();
double[] y_offS            = thetaS.Select(x => b/(Math.Sqrt(1 + (Math.Pow(a,2)/Math.Pow(b,2))*Math.Pow((Math.Tan(x)),2)))).ToArray();
double[] qS                = new double[] {0.5};
bool[] convectionS = new bool[] {false}; // convection

//BERECHNUNG
int ii = 0;
foreach(double theta in thetaS){
foreach(double y_off in y_offS){
foreach(double q in qS){
foreach(bool convection in convectionS){
foreach(int pDeg in degS){
foreach(IGridInfo grd in GridSeq){                                            
    //XNSE_Control C = Droplet_forWorksheet(true); // ICH HABE DIE WESENTLICHEN EINSTELLUNGEN HIER DIREKT IN DAS WORKSHEET ALS STATISCHE METHODE GESCHRIEBEN!
    XNSFE_Control C = Droplet_forWorksheet(true); // ICH HABE DIE WESENTLICHEN EINSTELLUNGEN HIER DIREKT IN DAS WORKSHEET ALS STATISCHE METHODE GESCHRIEBEN!

    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("theta", theta.ToString("N4")));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("heatflux", q));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("convection", convection));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("degree", pDeg));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("gridlevel", ((GridCommons)grd).Name.Split('_').Last()));

    C.SetDGdegree(pDeg);        
    C.SetGrid(grd);

    C.PhysicalParameters.rho_A             = 1.0;
    C.PhysicalParameters.rho_B             = 0.1; // unequal density
    C.PhysicalParameters.mu_A              = 0.5;
    C.PhysicalParameters.mu_B              = 0.05;
    C.PhysicalParameters.Sigma             = 0.1;
    C.PhysicalParameters.IncludeConvection = convection; // true;

    C.PhysicalParameters.betaS_A = 0.0;
    C.PhysicalParameters.betaS_B = 0.0;
    C.PhysicalParameters.betaL   = 0.0;
    C.PhysicalParameters.theta_e = theta;
    C.PhysicalParameters.Material   = false;   // SET TO false WHEN hvap != 0.0

    // 0.0 - no slip
    // positive number - some slip
    // double.PositiveInfinity - freeslip
    C.PhysicalParameters.slipI = 0.0;

    C.ThermalParameters.rho_A = C.PhysicalParameters.rho_A;
    C.ThermalParameters.rho_B = C.PhysicalParameters.rho_B;
    C.ThermalParameters.k_A = 1.0;
    C.ThermalParameters.k_B = 0.1;
    C.ThermalParameters.hVap = 0.0; // 0.0 means no evaporation -  for reference case!
    C.ThermalParameters.T_sat = 0.0;

    C.ThermalParameters.IncludeConvection = convection;

    C.LinearSolver                  = LinearSolverCode.direct_mumps.GetConfig();
    //C.LinearSolver                  = LinearSolverCode.direct_pardiso.GetConfig();
    //C.FailOnSolverFail = false;

    //VERDAMPFUNG IN MEDIUM A
    C.AddBoundaryValue("NavierSlip_linear_ConstantHeatflux");
    C.AddBoundaryValue("NavierSlip_linear_ConstantTemperature", "Temperature#B", "X => X[0]", false);


    C.AddInitialValue("Phi", $"X => ((X[0]).Pow2() /({a}).Pow2() + (X[1]+({y_off})).Pow2() / ({b}).Pow2()) - 1.0", false);
    C.FieldOptions[BoSSS.Solution.NSECommon.VariableNames.LevelSetCGidx(1)].Degree = 2;
    C.CutCellQuadratureType = XQuadFactoryHelper.MomentFittingVariants.Saye;
    C.AgglomerationThreshold = 0.3;

    // MAKE TIMESTEPPING SETTINGS MORE CLEAR!
    C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;

    C.SessionName = ((GridCommons)grd).Name + //grid
        "_P" + pDeg +// degree
        "_CA" + theta.ToString("N2") + // contact angle
        "_HF" + q.ToString("N2") + // heatflux
        "_C" + convection + "_Agg0.3"; // convection 

    // C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.MovingContactLineLogging());
    //controls[iDeg - iDeg0,iGrd - iGrd0] = C;
    controls.Add(C);
    //C.CreateJob(); // job will be permanently linked to control object
    Console.WriteLine($"{ii} : " + C.SessionName);
    ii++;                                        
}
}
}
}
}
}

0 : SlipDropletOnWall_Reference_meshStudy_4_P2_CA1.57_HF0.50_CFalse_Agg0.3
1 : SlipDropletOnWall_Reference_meshStudy_8_P2_CA1.57_HF0.50_CFalse_Agg0.3
2 : SlipDropletOnWall_Reference_meshStudy_16_P2_CA1.57_HF0.50_CFalse_Agg0.3
3 : SlipDropletOnWall_Reference_meshStudy_32_P2_CA1.57_HF0.50_CFalse_Agg0.3
4 : SlipDropletOnWall_Reference_meshStudy_64_P2_CA1.57_HF0.50_CFalse_Agg0.3
5 : SlipDropletOnWall_Reference_meshStudy_128_P2_CA1.57_HF0.50_CFalse_Agg0.3
6 : SlipDropletOnWall_Reference_meshStudy_4_P3_CA1.57_HF0.50_CFalse_Agg0.3
7 : SlipDropletOnWall_Reference_meshStudy_8_P3_CA1.57_HF0.50_CFalse_Agg0.3
8 : SlipDropletOnWall_Reference_meshStudy_16_P3_CA1.57_HF0.50_CFalse_Agg0.3
9 : SlipDropletOnWall_Reference_meshStudy_32_P3_CA1.57_HF0.50_CFalse_Agg0.3
10 : SlipDropletOnWall_Reference_meshStudy_64_P3_CA1.57_HF0.50_CFalse_Agg0.3
11 : SlipDropletOnWall_Reference_meshStudy_128_P3_CA1.57_HF0.50_CFalse_Agg0.3
12 : SlipDropletOnWall_Reference_meshStudy_4_P4_CA1.57_HF0.50_CFalse_Agg0.3
13 : SlipDro

In [133]:
// // FOR TESTING IF THIS RUNS
// foreach(var C in controls){
//var C = controls.ElementAt(0);
//C.ImmediatePlotPeriod = 1;
//C.SuperSampling = 2;
//C.savetodb = false;
//C.PostprocessingModules.Clear();
//using(var solver = new XNSFE()){
//       solver.Init(C);
//     solver.RunSolverMode();
//}
// }


In [134]:
controls.Count

18

In [135]:
bool run      = true;
var bpc = BoSSSshell.GetDefaultQueue();
bpc.DeployRuntime = true;

In [136]:
if(BoSSSshell.WorkflowMgm.AllJobs.Count() > 0){
     BoSSSshell.WorkflowMgm.ResetProject();
}
var jobs = controls.Select(c => c.CreateJob()).ToArray();

Not deleting any sessions, because not specified (`deleteSessions:false`).
Not deleting any grids, because not specified (`deleteGrids:false`).
Not deleting any grids, because not specified (`deleteDeployments:false`).
Forgetting all Jobs defined in this notebook so far...


In [137]:
jobs.ActivateTasks((MsHPC2012Client)bpc);

Deploying executables and additional files...
   copied 'win\amd64' runtime.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


...

   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


   written file: control.obj



In [14]:
// if(run){
//     jobs.Activate(bpc);
// } else {
//     foreach(var job in jobs) {
//         Console.WriteLine("Status Session: {0}", job.GetControl().SessionName);
//         var jS = job.Status;
//         Console.WriteLine(jS);        
//     }
// }

 Using the DeployAssembliesOnce option, this is experimental and untested if all necessary files are copied in all cases!
Deploying executables and additional files ... once
   copied 'win\amd64' runtime.
Deployments so far (2): (Job token: unknown, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver-binaries-2024Jan24_175354' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful), (Job token: 5068, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver2024Jan24_175408.493093' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful);
Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_4_P2_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:54:27	b4371fc5..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (2): (Job token: unknown, FinishedSuccessful 'SlipConver

Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_128_P3_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:56:43	f5f8c1e8..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (2): (Job token: unknown, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver-binaries-2024Jan24_175354' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful), (Job token: 5080, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver2024Jan24_175632.909412' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful);
Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_4_P4_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:56:45	393c313d..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedS

Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_64_P2_WF0.00_CA1.40_SL0.00_HF0.10_ST0.10_RTrue_CTrue	01/24/2024 18:35:31	a69feeea..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (3): (Job token: unknown, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver-binaries-2024Jan24_182332' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful), (Job token: unknown, Unknown 'SlipConvergence_Droplet-XNSFE_Solver-binaries-2024Jan24_182332' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, Unknown), (Job token: 5091, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver2024Jan24_183025.678803' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful);
Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy

Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_16_P4_WF0.00_CA1.40_SL0.00_HF0.10_ST0.10_RTrue_CTrue	01/24/2024 18:42:32	d001febb..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (2): (Job token: unknown, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver-binaries-2024Jan24_183628' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful), (Job token: 5101, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver2024Jan24_184232.853306' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful);
Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_32_P4_WF0.00_CA1.40_SL0.00_HF0.10_ST0.10_RTrue_CTrue	01/24/2024 18:42:47	fec4a41a..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedS

Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_16_P3_WF0.00_CA1.40_SL0.00_HF0.50_ST0.10_RTrue_CTrue	01/24/2024 19:03:23	acaac3f8..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (2): (Job token: unknown, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver-binaries-2024Jan24_185621' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful), (Job token: 5113, FinishedSuccessful 'SlipConvergence_Droplet-XNSFE_Solver2024Jan24_190324.403104' @ MS HPC client  Default @HPCCLUSTER4, @\\dc3\backup\rieckmann\cluster\binaries, FinishedSuccessful);
Success: 2
Info: Found successful session "SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_32_P3_WF0.00_CA1.40_SL0.00_HF0.50_ST0.10_RTrue_CTrue	01/24/2024 19:03:46	b2235e46..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedS

...

timeout waiting for job.exe submit /jobfile:"\\dc3\backup\rieckmann\cluster\binaries\SlipConvergence_Droplet-XNSFE_Solver2024Jan24_210913.558963\job.xml"  /scheduler:HPCCLUSTER4 /user:FDY\rieckmann 


Caught IOException : timeout waiting for job.exe submit /jobfile:"\\dc3\backup\rieckmann\cluster\binaries\SlipConvergence_Droplet-XNSFE_Solver2024Jan24_210913.558963\job.xml"  /scheduler:HPCCLUSTER4 /user:FDY\rieckmann 
Moving on ...

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job SlipDropletOnWall_meshStudy_4_P2_WF0.00_CA1.40_SLInfinity_HF0.00_ST0.10_RTrue_CTrue ... 


Error: System.ArgumentException: Unable to serialize/deserialize 'id:interfacesliplength' correctly. (Missing a DataMemberAtribute in control class?)
   at BoSSS.Application.BoSSSpad.AppControlExtensions.VerifyEx(AppControl ctrl) in D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\src\L4-application\BoSSSpad\AppControlExtensions.cs:line 531
   at BoSSS.Application.BoSSSpad.Job.FiddleControlFile(BatchProcessorClient bpc) in D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\src\L4-application\BoSSSpad\Job.cs:line 488
   at BoSSS.Application.BoSSSpad.Job.Reactivate() in D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\src\L4-application\BoSSSpad\Job.cs:line 1544
   at BoSSS.Application.BoSSSpad.Job.Activate(BatchProcessorClient bpc) in D:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-InterfaceSlip\public\src\L4-application\BoSSSpad\Job.cs:line 1487
   at BoSSS.Application.BoSSSpad.AppControlExtensions.Activate(IEnumerable`1 jobs, BatchProcessorClient BatchSys, Boolean D

In [18]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions.ForEach(s => display(s));

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_128_P2_WF0.00_CA1.40_SL0.10_HF0.50_ST0.10_RTrue_CTrue	01/25/2024 10:06:31	cea85c50...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_64_P4_WF0.00_CA1.40_SL0.10_HF0.10_ST0.10_RTrue_CTrue	01/25/2024 10:03:50	aebb2754...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_64_P2_WF0.00_CA1.40_SL0.10_HF0.50_ST0.10_RTrue_CTrue	01/25/2024 10:06:06	f5617a82...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_16_P3_WF0.00_CA1.40_SL0.10_HF0.50_ST0.10_RTrue_CTrue	01/25/2024 10:09:33	8b1aaf0e...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_8_P3_WF0.00_CA1.40_SL0.10_HF0.50_ST0.10_RTrue_CTrue	01/25/2024 10:09:18	906604c5...

...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_8_P3_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:55:35	8faee681...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_4_P3_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:55:20	eed7b59a...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_16_P2_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:54:39	7b8f702a...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_8_P2_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:54:29	86e71ece...

SlipConvergence_Droplet	SlipDropletOnWall_meshStudy_4_P2_WF0.00_CA1.40_SL0.00_HF0.00_ST0.10_RTrue_CTrue	01/24/2024 17:54:27	b4371fc5...

In [19]:
sessions.Count()

200

In [ ]:
// Für Vislt notwendig

sessions.ForEach(session=> session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do());
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).WithShadowFields().Do();
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do();

Starting export process... Data will be written to the directory: p:\Bachelorthesis_Pasic\Kickoff\BoSSS\DropletOnSlipWall\02_Worksheets\SlipConvergence_Droplet_SlipInterface_with_HeatFlux\SlipDropletOnWall_90Deg_beta0_cAstat90_roh_A1_roh_B0.1_lsInfinity_q-0.5_ConvStudy2_k4_mesh5
Starting export process... Data will be written to the directory: p:\Bachelorthesis_Pasic\Kickoff\BoSSS\DropletOnSlipWall\02_Worksheets\SlipConvergence_Droplet_SlipInterface_with_HeatFlux\SlipDropletOnWall_90Deg_beta0_cAstat90_roh_A1_roh_B0.1_lsInfinity_q-0.1_ConvStudy2_k4_mesh5
Starting export process... Data will be written to the directory: p:\Bachelorthesis_Pasic\Kickoff\BoSSS\DropletOnSlipWall\02_Worksheets\SlipConvergence_Droplet_SlipInterface_with_HeatFlux\SlipDropletOnWall_90Deg_beta0_cAstat90_roh_A1_roh_B0.1_lsInfinity_q-0.5_ConvStudy2_k3_mesh5
Starting export process... Data will be written to the directory: p:\Bachelorthesis_Pasic\Kickoff\BoSSS\DropletOnSlipWall\02_Worksheets\SlipConvergence_Droplet_

In [ ]:
// Process.Start("explorer.exe", sessions.Third().GetSessionDirectory());